In [ ]:
# ============================================================
# CELL 1 — LOAD DATA + CHUNK KNOWLEDGE BASE
# Saves chunks and combined test set to Drive
# ============================================================

!pip install -q sentence-transformers faiss-cpu rank-bm25 \
    transformers accelerate bitsandbytes sacrebleu rapidfuzz \
    bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, re, pickle, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
import torch

OUT = Path("/content/drive/MyDrive/Govt_Chatbot/RAG")
OUT.mkdir(parents=True, exist_ok=True)

# ---------- Find uploaded files automatically ----------
def find_file(pattern):
    files = glob.glob("/content/" + pattern)
    if not files:
        raise FileNotFoundError(pattern)
    return max(files, key=os.path.getmtime)

KB_FILES = {
    "passport": find_file("passport_clean_kb*.json"),
    "nid": find_file("NID_clean_kb*.json"),
    "tin": find_file("TIN_clean_kb*.json"),
    "birth_death": find_file("birth_death_clean_kb*.json"),
}

TEST_FILES = {
    "passport": find_file("passport_qa_test*.json"),
    "nid": find_file("nid_qa_test*.json"),
    "tin": find_file("TIN_qa_test*.json"),
    "birth_death": find_file("birth_death_qa_test*.json"),
}

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def norm_domain(x):
    x = str(x).lower()
    if "passport" in x:
        return "passport"
    if "birth" in x or "death" in x:
        return "birth_death"
    if "nid" in x:
        return "nid"
    if "tin" in x:
        return "tin"
    return x

# Simple word-based chunking
def chunk_text(text, size=180, overlap=30):
    words = str(text).split()

    if len(words) <= size:
        return [str(text).strip()]

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap

    return chunks

# ---------- Build corpus ----------
chunks = []

for domain, path in KB_FILES.items():

    for row in load_json(path):

        text = str(row.get("text", "")).strip()
        if not text:
            continue

        doc_id = str(
            row.get("doc_id", f"{domain}_{len(chunks)}")
        )

        for part, text_chunk in enumerate(chunk_text(text)):

            title = str(row.get("title", "")).strip()
            topic = str(row.get("topic", "")).strip()

            # Text used by retrieval
            index_text = (
                f"শিরোনাম: {title}\n"
                f"বিষয়: {topic}\n"
                f"তথ্য: {text_chunk}"
            )

            chunks.append({
                "chunk_id": len(chunks),
                "doc_id": doc_id,
                "domain": norm_domain(row.get("domain", domain)),
                "title": title,
                "topic": topic,
                "source_url": str(row.get("source_url", "")),
                "text": text_chunk,
                "index_text": index_text
            })

# ---------- Combine test questions ----------
tests = []

for domain, path in TEST_FILES.items():

    for row in load_json(path):

        tests.append({
            "id": str(row.get("id", "")),
            "domain": norm_domain(row.get("domain", domain)),
            "question": str(row.get("instruction", "")).strip(),
            "gold": str(row.get("output", "")).strip()
        })

# Save
with open(OUT / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

pd.DataFrame(tests).to_csv(
    OUT / "test_questions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("KB files:")
for k, v in KB_FILES.items():
    print(k, "->", os.path.basename(v))

print("\nTotal chunks:", len(chunks))
print("Total test questions:", len(tests))
print("Saved to:", OUT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.6 MB/s eta 0:00:00
Mounted at /content/drive
KB files:
passport -> passport_clean_kb(2).json
nid -> NID_clean_kb(1).json
tin -> TIN_clean_kb(1).json
birth_death -> birth_death_clean_kb(1).json

Total chunks: 1284
Total test questions: 248
Saved to: /content/drive/MyDrive/Govt_Chatbot/RAG


In [ ]:
# ============================================================
# CELL 2 — BUILD AND STORE DENSE + SPARSE INDICES
# BGE-M3 -> FAISS
# BM25   -> Pickle
# ============================================================

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import gc

with open(OUT / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [x["index_text"] for x in chunks]

# ---------- BGE-M3 dense embeddings ----------
device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    "BAAI/bge-m3",
    device=device
)

embeddings = embedder.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

# FAISS inner product = cosine similarity because vectors normalized
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(
    index,
    str(OUT / "bge_m3.faiss")
)

np.save(
    OUT / "bge_m3_embeddings.npy",
    embeddings
)

# ---------- BM25 ----------
BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def tokenize(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BN_TO_EN).lower()

    return re.findall(
        r"[\u0980-\u09FF]+|[a-z]+|\d+(?:\.\d+)?",
        text
    )

tokenized_corpus = [tokenize(x) for x in texts]

bm25 = BM25Okapi(tokenized_corpus)

with open(OUT / "bm25.pkl", "wb") as f:
    pickle.dump({
        "bm25": bm25,
        "tokens": tokenized_corpus
    }, f)

print("FAISS vectors:", index.ntotal)
print("BM25 documents:", bm25.corpus_size)

# Free GPU memory
del embedder, embeddings
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Indices saved successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

FAISS vectors: 1284
BM25 documents: 1284
Indices saved successfully.


In [ ]:
# ============================================================
# CELL 3 — HYBRID RETRIEVAL + RERANKING + QWEN
# BGE-M3 + BM25 -> RRF -> BGE Reranker -> Top-5 -> Qwen
# Saves retrievals.csv, retrievals.json and predictions.csv
# ============================================================

import json, pickle, gc, re, unicodedata
import numpy as np
import pandas as pd
import torch
import faiss

from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

OUT = Path("/content/drive/MyDrive/Govt_Chatbot/RAG")

# ------------------------------------------------------------
# Load chunks, questions and stored indices
# ------------------------------------------------------------

with open(OUT / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

tests = pd.read_csv(
    OUT / "test_questions.csv"
).fillna("")

faiss_index = faiss.read_index(
    str(OUT / "bge_m3.faiss")
)

with open(OUT / "bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)["bm25"]


BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def tokenize(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BN_TO_EN).lower()

    return re.findall(
        r"[\u0980-\u09FF]+|[a-z]+|\d+(?:\.\d+)?",
        text
    )


# ============================================================
# 1. LOAD BGE-M3 + RERANKER
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    "BAAI/bge-m3",
    device=device
)

reranker_name = "BAAI/bge-reranker-v2-m3"

rerank_tokenizer = AutoTokenizer.from_pretrained(
    reranker_name
)

reranker = AutoModelForSequenceClassification.from_pretrained(
    reranker_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

reranker.eval()


# ============================================================
# 2. HYBRID RETRIEVAL
# ============================================================

def hybrid_retrieve(question, domain, candidate_k=25):

    # ---------- Dense ----------
    qvec = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    dense_scores, dense_ids = faiss_index.search(
        qvec,
        faiss_index.ntotal
    )

    dense_rank = []

    for score, idx in zip(
        dense_scores[0],
        dense_ids[0]
    ):
        idx = int(idx)

        if chunks[idx]["domain"] == domain:
            dense_rank.append((idx, float(score)))

        if len(dense_rank) >= 100:
            break


    # ---------- BM25 ----------
    bm25_scores = bm25.get_scores(
        tokenize(question)
    )

    sparse_ids = np.argsort(
        bm25_scores
    )[::-1]

    sparse_rank = []

    for idx in sparse_ids:

        idx = int(idx)

        if chunks[idx]["domain"] == domain:

            sparse_rank.append(
                (
                    idx,
                    float(bm25_scores[idx])
                )
            )

        if len(sparse_rank) >= 100:
            break


    # ---------- Reciprocal Rank Fusion ----------
    rrf = defaultdict(float)

    dense_map = {}
    bm25_map = {}

    for rank, (idx, score) in enumerate(
        dense_rank,
        1
    ):
        rrf[idx] += 1 / (60 + rank)
        dense_map[idx] = score

    for rank, (idx, score) in enumerate(
        sparse_rank,
        1
    ):
        rrf[idx] += 1 / (60 + rank)
        bm25_map[idx] = score


    best_ids = sorted(
        rrf,
        key=rrf.get,
        reverse=True
    )[:candidate_k]


    candidates = []

    for idx in best_ids:

        item = chunks[idx].copy()

        item["dense_score"] = dense_map.get(idx, 0)
        item["bm25_score"] = bm25_map.get(idx, 0)
        item["rrf_score"] = rrf[idx]

        candidates.append(item)

    return candidates


# ============================================================
# 3. RERANK TOP-25 -> FINAL TOP-5
# ============================================================

def rerank(question, candidates, top_k=5):

    scores = []

    for start in range(0, len(candidates), 4):

        batch = candidates[start:start + 4]

        encoded = rerank_tokenizer(
            [question] * len(batch),
            [x["index_text"] for x in batch],
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():

            logits = reranker(
                **encoded
            ).logits.squeeze(-1)

        scores.extend(
            logits.float().cpu().tolist()
        )


    for item, score in zip(
        candidates,
        scores
    ):
        item["reranker_score"] = float(score)


    return sorted(
        candidates,
        key=lambda x: x["reranker_score"],
        reverse=True
    )[:top_k]


# ============================================================
# 4. RETRIEVE ALL TEST QUESTIONS
# ============================================================

retrieval_cache = []
retrieval_rows = []

for _, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Hybrid retrieval + reranking"
):

    candidates = hybrid_retrieve(
        row["question"],
        row["domain"]
    )

    docs = rerank(
        row["question"],
        candidates,
        top_k=5
    )

    retrieval_cache.append(docs)

    save_row = {
        "id": row["id"],
        "domain": row["domain"],
        "question": row["question"]
    }

    for i, doc in enumerate(docs, 1):

        save_row[f"doc_{i}"] = doc["doc_id"]
        save_row[f"title_{i}"] = doc["title"]
        save_row[f"context_{i}"] = doc["text"]
        save_row[f"source_{i}"] = doc["source_url"]

        save_row[f"dense_score_{i}"] = doc["dense_score"]
        save_row[f"bm25_score_{i}"] = doc["bm25_score"]
        save_row[f"rrf_score_{i}"] = doc["rrf_score"]
        save_row[f"reranker_score_{i}"] = doc["reranker_score"]


pd.DataFrame(
    retrieval_rows if False else []  # harmless placeholder
)

# rebuild rows cleanly
retrieval_rows = []

for row_idx, row in tests.iterrows():

    docs = retrieval_cache[row_idx]

    x = {
        "id": row["id"],
        "domain": row["domain"],
        "question": row["question"]
    }

    for i, doc in enumerate(docs, 1):

        x[f"doc_{i}"] = doc["doc_id"]
        x[f"title_{i}"] = doc["title"]
        x[f"context_{i}"] = doc["text"]
        x[f"source_{i}"] = doc["source_url"]
        x[f"dense_score_{i}"] = doc["dense_score"]
        x[f"bm25_score_{i}"] = doc["bm25_score"]
        x[f"rrf_score_{i}"] = doc["rrf_score"]
        x[f"reranker_score_{i}"] = doc["reranker_score"]

    retrieval_rows.append(x)


pd.DataFrame(retrieval_rows).to_csv(
    OUT / "retrievals.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    OUT / "retrievals.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        retrieval_cache,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Retrievals saved.")


# ============================================================
# 5. FREE RETRIEVAL MODELS BEFORE QWEN
# ============================================================

del embedder
del reranker
del rerank_tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 6. LOAD QWEN-2.5-7B-INSTRUCT
# ============================================================

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer_qwen = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto"
)

model.eval()

model_device = (
    model
    .get_input_embeddings()
    .weight
    .device
)

MAX_NEW_TOKENS = 450


# ============================================================
# 7. GENERATE ANSWER
# ============================================================

def generate_answer(question, docs):

    context = "\n\n".join([
        f"[Context {i}]\n"
        f"শিরোনাম: {doc['title']}\n"
        f"তথ্য: {doc['text']}"
        for i, doc in enumerate(docs, 1)
    ])

    messages = [
        {
            "role": "system",
            "content":
                "তুমি বাংলাদেশের সরকারি সেবা বিষয়ক সহকারী। "
                "শুধু প্রদত্ত Context ব্যবহার করে বাংলায় সরাসরি উত্তর দাও। "
                "Context-এর ভাষা প্রশ্নের থেকে আলাদা হলেও সমার্থক তথ্য বুঝে উত্তর দাও। "
                "ফি, সংখ্যা, সময় ও প্রয়োজনীয় কাগজপত্র নির্ভুলভাবে উল্লেখ করো। "
                "অপ্রয়োজনীয় ব্যাখ্যা দিও না। "
                "Context-এ উত্তর একেবারেই না থাকলে শুধু বলো: "
                "'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
        },
        {
            "role": "user",
            "content":
                f"Context:\n{context}\n\n"
                f"প্রশ্ন: {question}\n\n"
                "উত্তর:"
        }
    ]

    prompt = tokenizer_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_qwen(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to(model_device)

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer_qwen.eos_token_id,
            eos_token_id=tokenizer_qwen.eos_token_id
        )

    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer_qwen.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and
        generated[-1].item()
        != tokenizer_qwen.eos_token_id
    )

    return answer, truncated


# ============================================================
# 8. GENERATE + CHECKPOINT
# ============================================================

predictions = []

for i, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Qwen generation"
):

    answer, truncated = generate_answer(
        row["question"],
        retrieval_cache[i]
    )

    predictions.append({
        "id": row["id"],
        "domain": row["domain"],
        "question": row["question"],
        "gold": row["gold"],
        "prediction": answer,
        "truncated": truncated
    })

    # Save after every question
    pd.DataFrame(predictions).to_csv(
        OUT / "predictions_partial.csv",
        index=False,
        encoding="utf-8-sig"
    )


pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCompleted:", len(pred_df))
print("Truncated:", int(pred_df["truncated"].sum()))
print("Saved:", OUT / "retrievals.csv")
print("Saved:", OUT / "predictions.csv")


del model, tokenizer_qwen
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen generation:   0%|          | 0/248 [00:00<?, ?it/s]


Completed: 248
Truncated: 13
Saved: /content/drive/MyDrive/Govt_Chatbot/RAG/retrievals.csv
Saved: /content/drive/MyDrive/Govt_Chatbot/RAG/predictions.csv


In [4]:
# ============================================================
# CELL 4 — FINAL EVALUATION
# Exact Match, Fuzzy, BLEU, ROUGE-1/2/L,
# Token F1 and multilingual BERTScore
# Saves predictions.csv + result.csv
# ============================================================

import re
import unicodedata
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score

OUT = Path(
    "/content/drive/MyDrive/Govt_Chatbot/RAG"
)

df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")


# ============================================================
# TEXT NORMALIZATION
# ============================================================

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ============================================================
# EXACT MATCH
# ============================================================

def exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ============================================================
# TOKEN F1
# ============================================================

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (
            Counter(p)
            &
            Counter(g)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# ROUGE-N
# ============================================================

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pg = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gg = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pg & gg).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pg.values())
    recall = overlap / sum(gg.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# ROUGE-L
# ============================================================

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# CALCULATE ROW METRICS
# ============================================================

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_texts = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_texts = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_texts,
        [gold_texts]
    ).score
    / 100
)


# ============================================================
# BERTSCORE
# ============================================================

print("Calculating multilingual BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ============================================================
# FINAL RESULT TABLE
# ============================================================

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1",
        "Truncated Outputs"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean(),
        (
            df["truncated"]
            .astype(str)
            .str.lower()
            .eq("true")
            .sum()
        )
    ]
})


# Save row-level metrics
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save final result
result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(OUT / "retrievals.csv")
print(OUT / "retrievals.json")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating multilingual BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 44.81 seconds, 5.53 sentences/sec


,metric,score
0,Exact Match,0.169355
1,Fuzzy Match,0.881228
2,Corpus BLEU,0.412181
3,ROUGE-1,0.653824
4,ROUGE-2,0.563022
5,ROUGE-L,0.621993
6,Token F1,0.653824
7,BERT Precision,0.843939
8,BERT Recall,0.871810
9,BERT F1,0.856216



Saved:
/content/drive/MyDrive/Govt_Chatbot/RAG/retrievals.csv
/content/drive/MyDrive/Govt_Chatbot/RAG/retrievals.json
/content/drive/MyDrive/Govt_Chatbot/RAG/predictions.csv
/content/drive/MyDrive/Govt_Chatbot/RAG/result.csv
